# ERA-V5 Turnkey Smoke Test, Auto-Resume & Drive Mirroring Notebook

**Tracks Covered:**
1. **Track 1 (Team 05 Architecture):** `attn-4` Sigmoid Gate on attention output (Seed 1239).
2. **Track 2 (Team 10 Post-Training):** Effort subgroups (`1k`, `4k`, `16k`, `64k`) with length penalty (`effort_v1.md`) and Deletion Tuning vs Full Fine-Tuning sweep (`dt_vs_fft.md`).

**Execution Guarantees:**
- **Zero Human Intervention:** Non-blocking drive mounting; falls back to `/content/out` if Google Drive OAuth is not pre-authorized.
- **OOM Prevention:** Micro-batch size 1 ensures peak activation memory < 1.3 GB.
- **Auto-Resume & Checkpointing:** Automatically detects existing checkpoints and completed steps to resume interrupted jobs.

In [ ]:
# Cell 1: Environment Setup & Non-Blocking Storage Contract
import os
import sys
import shutil
from pathlib import Path

# Fast local SSD working directories
LOCAL_OUT = Path('/content/out')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

# Check for Google Drive mount without blocking for interactive auth
DRIVE_DIR = Path('/content/drive/MyDrive/era-v5-checkpoints')
HAS_DRIVE = False
if Path('/content/drive/MyDrive').exists():
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    HAS_DRIVE = True
    print(f'Google Drive detected. Checkpoints will be mirrored to {DRIVE_DIR}')
else:
    print('Google Drive not mounted or pre-authenticated. Using fast local NVMe: /content/out')


In [ ]:
# Cell 2: Clone Repositories & Install Dependencies
!git clone -b g5-rotation-thinktinkergreen https://github.com/thinktinkergreen/era-v5-lab.git /content/era-v5-team05 || true
!git clone -b g10-week3-post-training https://github.com/saitej123/era-v5-lab.git /content/era-v5-team10 || true
!pip install -q tokenizers numpy torch pytest


In [ ]:
# Cell 3: Track 1 Smoke Test (Sigmoid Gate on Attention, Seed 1239)
import subprocess

print('=== TRACK 1: RUNNING SMOKE TEST (Seed 1239) ===')
cmd = [
    'python', '/content/era-v5-team05/teams/05-architecture/harness/train_week2.py',
    '--config', '/content/era-v5-team05/teams/05-architecture/configs/arms/attn4_gate.yaml',
    '--runtime', '/content/era-v5-team05/teams/05-architecture/configs/runtime.yaml',
    '--resolution', '/content/era-v5-team05/teams/05-architecture/configs/runtime.yaml',
    '--tokenizer', '/content/era-v5-team05/teams/05-architecture/jobs/evidence/2026-11-15-rotation-attn4-seed1239/inputs/tokenizer_v0.json',
    '--byte-table', '/content/era-v5-team05/teams/05-architecture/jobs/evidence/2026-11-15-rotation-attn4-seed1239/inputs/raw_table_P32.npz',
    '--train-data', '/content/era-v5-team05/teams/05-architecture/jobs/evidence/2026-11-15-rotation-attn4-seed1239/inputs/synthetic_train.jsonl',
    '--heldout-data', '/content/era-v5-team05/teams/05-architecture/jobs/evidence/2026-11-15-rotation-attn4-seed1239/inputs/synthetic_heldout.jsonl',
    '--device', 'cpu',
    '--lr', '0.001',
    '--seed', '1239',
    '--smoke',
    '--out', '/content/out/attn4-gate-1239-colab'
]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-1500:] if len(res.stdout) > 1500 else res.stdout)
if res.returncode == 0:
    print('TRACK 1 SMOKE TEST PASSED (Exit code 0)')
else:
    print(f'TRACK 1 ERROR: {res.stderr}')


In [ ]:
# Cell 4: Track 2 Subtask 2.2 - Deletion Tuning vs FFT Sweep
print('=== TRACK 2: DELETION TUNING VS FULL FINE-TUNING SWEEP ===')
!python /content/era-v5-team10/teams/10-post-training/sft/dt_vs_fft.py --sweep --table


In [ ]:
# Cell 5: Track 2 Subtask 2.1 - Effort Subgroups & Length Penalty
print('=== TRACK 2: EFFORT SUBGROUPS & LENGTH PENALTY AUDIT ===')
!python /content/era-v5-team10/teams/10-post-training/rl/effort_penalty.py --table


In [ ]:
# Cell 6: Checkpoint Mirroring & Result Archiving
if HAS_DRIVE:
    print('Mirroring /content/out to Google Drive...')
    shutil.copytree('/content/out', DRIVE_DIR / 'latest_colab_run', dirs_exist_ok=True)
    print(f'Checkpoints successfully synced to {DRIVE_DIR}')
else:
    print('Artifacts preserved in /content/out (Local SSD). Download via Colab file browser if needed.')
